# PLANTILLA TRANSFORMACIÓN DE DATOS

**IMPORTANTE**: Recuerda hacer una copia de esta plantilla para no machacar la original.

## IMPORTAR PAQUETES

In [75]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from category_encoders import TargetEncoder
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import Binarizer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import QuantileTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

#Automcompletar rápido
%config IPCompleter.greedy=True

## IMPORTAR LOS DATOS

1.- Sustituir la ruta del proyecto.

In [76]:
ruta_proyecto = 'C:/Users/sergi/Desktop/DATA/CURSO DS4B/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/01_LEADSCORING'

2.- Nombrar los ficheros de datos.

In [82]:
nombre_cat = 'cat_resultado_eda.pickle'
nombre_num = 'num_resultado_eda.pickle'

3.- Cargar los datos.

In [84]:
cat = pd.read_pickle(ruta_proyecto + '/02_Datos/03_Trabajo/' + nombre_cat).reset_index(drop = True)
num = pd.read_pickle(ruta_proyecto + '/02_Datos/03_Trabajo/' + nombre_num)

4.- Separar la target.

In [85]:
target = num[['compra']].copy().reset_index(drop=True)

In [86]:
num.columns

Index(['compra', 'visitas_total', 'tiempo_en_site_total',
       'paginas_vistas_visita', 'score_actividad', 'score_perfil'],
      dtype='object')

## TRANSFORMACIÓN DE CATEGÓRICAS

### One Hot Encoding

#### Variables a aplicar OHE

In [87]:
var_ohe = ['origen','fuente','ult_actividad','ambito','ocupacion','descarga_lm']

#### Instanciar

In [88]:
ohe = OneHotEncoder(sparse = False, handle_unknown='ignore')

#### Entrenar y aplicar

In [89]:
cat_ohe = ohe.fit_transform(cat[var_ohe])

C:\Users\sergi\miniconda3\envs\proyecto1\Lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


#### Guardar como dataframe

In [90]:
cat_ohe = pd.DataFrame(cat_ohe, columns = ohe.get_feature_names_out())

## TRANSFORMACIÓN DE NUMÉRICAS

## UNIFICAR DATASETS TRANSFORMADOS

In [93]:
df = pd.concat([cat_ohe,num.reset_index()], axis = 1)

## REESCALAR VARIABLES

### Con Min-Max

In [94]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4792 entries, 0 to 4791
Data columns (total 43 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   origen_API                                4792 non-null   float64
 1   origen_Landing Page Submission            4792 non-null   float64
 2   origen_Lead Add Form                      4792 non-null   float64
 3   origen_OTROS                              4792 non-null   float64
 4   fuente_Chat                               4792 non-null   float64
 5   fuente_Direct Traffic                     4792 non-null   float64
 6   fuente_Google                             4792 non-null   float64
 7   fuente_OTROS                              4792 non-null   float64
 8   fuente_Organic Search                     4792 non-null   float64
 9   fuente_Reference                          4792 non-null   float64
 10  ult_actividad_Chat Conversation     

#### Variables a reescalar con Min-Max

In [95]:
var_mms = df.iloc[:,39:].columns

#### Instanciar

In [96]:
mms = MinMaxScaler()

#### Entrenar y aplicar

In [97]:
df_mms = mms.fit_transform(df[var_mms])

#### Guardar como dataframe

In [98]:
#Añadir sufijos a los nombres
nombres_mms = [variable + '_mms' for variable in var_mms]

#Guardar como dataframe
df_mms = pd.DataFrame(df_mms,columns = nombres_mms)

## UNIFICAR DATASETS REESCALADOS

In [99]:
id = df.id

### Crear una lista con los dataframes a incluir en el tablón analítico

In [101]:
df_tablon = pd.concat([id,cat_ohe,df_mms,target], axis = 1).set_index('id')

## GUARDAR DATASET TRAS TRANSFORMACIÓN DE DATOS

En formato pickle para no perder las modificaciones de metadatos.

In [102]:
#Definir los nombres del archivo
ruta_df_tablon = ruta_proyecto + '/02_Datos/03_Trabajo/' + 'df_tablon.pickle'

In [103]:
#Guardar los archivos
df_tablon.to_pickle(ruta_df_tablon)